In [1]:
import dash
import dash_core_components as dcc
import dash_html_components as html
import pandas as pd
import plotly.express as px
from dash.dependencies import Input, Output
import dash_bootstrap_components as dbc


C:\Users\NoLo\AppData\Local\Temp\ipykernel_7468\1030810081.py:2: UserWarning: 
The dash_core_components package is deprecated. Please replace
`import dash_core_components as dcc` with `from dash import dcc`
  import dash_core_components as dcc
C:\Users\NoLo\AppData\Local\Temp\ipykernel_7468\1030810081.py:3: UserWarning: 
The dash_html_components package is deprecated. Please replace
`import dash_html_components as html` with `from dash import html`
  import dash_html_components as html


In [2]:
# ---------------------------
# 1. Загрузка и подготовка данных
# ---------------------------
players = pd.read_csv('players.csv')
matches = pd.read_csv('matches.csv')
champions = pd.read_csv('champions.csv')
# Определяем регион по префиксу match_id
matches['region'] = matches['match_id'].apply(lambda x: 'americas' if x.startswith('NA1_') else 'europe')
# Преобразуем длительность в минуты
matches['duration_min'] = matches['game_duration_sec'] / 60
# Для удобства переведём win в числовой формат (0/1)
matches['win_num'] = matches['win'].astype(int)
matches = matches.merge(champions, on= 'champion')

In [3]:
region_selector = dcc.Dropdown(
      id= 'region_selector',
      options= [{'label': r, 'value': r} for r in matches['region'].unique()],
      value= ['europe', 'americas'],
      multi= True
)

In [4]:
# ---------------------------
# 3. Создание Dash-приложения
# ---------------------------
app = dash.Dash(
    external_stylesheets=[dbc.themes.BOOTSTRAP]
)
app.title = 'LoL Match Analytics'

app.layout = html.Div([
             dbc.Row(html.H1("Анализ матчей League of Legends")),
             dbc.Row([html.Div('Фильтр регионов'),
                      html.Div(region_selector)]),
             dbc.Row([
                 dbc.Col(dcc.Graph(id= 'wins_rate'),
                         width= {'size': 6}),
                 dbc.Col(dcc.Graph(id= 'histogram-lp'), 
                         width=6)]),
             dbc.Row([
                 dbc.Col(dcc.Graph(id= 'box_id'),
                         width= {'size': 6}),
                 dbc.Col(dcc.Graph(id= 'scatter_id'), 
                         width=6)])
    ],
             style= {'margin-left': '80px',
                     'margin-right': '80px'})

@app.callback(
    [Output('wins_rate', 'figure'),
     Output('histogram-lp', 'figure'),
     Output('box_id', 'figure'),
     Output('scatter_id', 'figure')],
    [Input('region_selector', 'value')]
)
def update_graph(selected_regions):
    chart_data= matches[matches['region'].isin(selected_regions)]
    chart_data_ply= players[players['region'].isin(selected_regions)]
    winrate_df = chart_data.groupby('champion')['win_num'].agg(['count', 'sum']).reset_index()
    winrate_df['win_rate'] = winrate_df['sum'] / winrate_df['count'] * 100
    winrate_df = winrate_df[winrate_df['count'] >= 1]  # фильтруем малоиграемых
    champ_f = chart_data.groupby('champion')[['kills', 'deaths']].mean().reset_index()
    dur_f = chart_data.groupby('match_id')['duration_min'].first()

    top_wr = winrate_df.nlargest(15, 'win_rate').sort_values('win_rate', ascending=True)
    # График 1: топ-10 чемпионов по винрейту
    fig_wr = px.bar(top_wr, x='win_rate', y='champion', orientation='h',
                    title=f'Топ-15 чемпионов по винрейту',
                    labels={'win_rate': 'Винрейт (%)', 'champion': 'Чемпион'},
                    text='win_rate', color='win_rate', color_continuous_scale='Viridis')
    fig_wr.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
    fig_wr.update_layout(xaxis_range=[0, 100])

    # График 2: гистограмма распределения "LP"
    fig_hist = px.histogram(chart_data_ply, x='league_points', nbins=30,
                            title=f'Распределение игроков по очкам лиги (LP)',
                            labels={'league_points': 'Количество LP', 'count': 'Количество'},
                            color_discrete_sequence=['#636EFA'],
                            marginal='box')
    fig_hist.update_layout(bargap=0.05)

    # График 3: диаграмма размаха длительности матчей
    fig_box = px.box(dur_f, y='duration_min', title=f'Длительность матчей',
                         labels={'duration_min': 'Длительность (мин)'})
    
    # График 4: scatter-plot (средние убийства vs смерти по чемпионам)
    fig_scatter = px.scatter(champ_f, x='kills', y='deaths',
                             title=f'Средние убийства и смерти по чемпионам)',
                             labels={'kills': 'Средние убийства', 'deaths': 'Средние смерти'},
                             hover_name=champ_f['champion'], trendline='ols')
    fig_scatter.update_traces(marker=dict(size=8, opacity=0.7))
    fig_scatter.update_traces(marker=dict(size=10, opacity=0.7, line=dict(width=1, color='DarkSlateGray')))
    
    return fig_wr, fig_hist, fig_box, fig_scatter

if __name__ == '__main__':
    app.run(debug=True)